Notebook contents (copy & paste into a new notebook)
# 4_VGG16_Transfer_Learning.ipynb
**Transfer Learning with VGG16 for Facial Emotion Recognition**

This notebook uses a pretrained VGG16 backbone (imagenet weights) and fine-tunes it for 7-class emotion classification:
`['Angry','Disgust','Fear','Happy','Neutral','Sad','Surprise']`

Steps:
1. Import libs
2. Set dataset path + ImageDataGenerator (80/20)
3. Build model (VGG16 base + custom head)
4. Train head (base frozen)
5. Unfreeze top VGG blocks and fine-tune
6. Evaluate and save
7. (Optional) Grad-CAM for saliency visualization

In [1]:
# 1) Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import tensorflow as tf

# print TF version and GPU availability (optional)
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU available: []


In [2]:
# 2) Dataset path and parameters — update if required
dataset_path = r"F:\\TERM 7\\CSM422 (DEEP LEARNING)\\Emotion_recognitition\\notebooks\\dataset"
img_size = (224, 224)   # VGG16 expects >= 48 but standard is 224x224
batch_size = 32
seed = 42

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset path not found: {dataset_path}")

In [3]:
# 3) Data generators with augmentation for training, simple rescale for validation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # use VGG preprocessing (scaling & mean subtraction)
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    seed=seed
)

val_gen = val_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,   # we will use `.classes` for evaluation; shuffle False keeps order
    seed=seed
)

num_classes = train_gen.num_classes
class_indices = train_gen.class_indices
print("Classes (index->label):", class_indices)

Found 31583 images belonging to 8 classes.
Found 7893 images belonging to 8 classes.
Classes (index->label): {'Angry': 0, 'Disgust': 1, 'Fear': 2, 'Happy': 3, 'Neutral': 4, 'Sad': 5, 'Surprise': 6, 'images': 7}


In [4]:
# 4) Build model: VGG16 base (without top) + custom classification head
# Load VGG16 base
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Freeze base model for initial training
base_model.trainable = False

# Build top layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Summary
model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 43s 1us/step 


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv1 (Conv2D)                │ (None, 224, 224, 64)        │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                │ (None, 224, 224, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)           │ (None, 112, 112, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                │ (None, 112, 112, 128)       │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                │ (None, 112, 112, 128)       │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)           │ (None, 56, 56, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                │ (None, 56, 56, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)           │ (None, 28, 28, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                │ (None, 28, 28, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)           │ (None, 14, 14, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)           │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 512)                 │           2,0

 Total params: 15,112,776 (57.65 MB)

 Trainable params: 397,064 (1.51 MB)

 Non-trainable params: 14,715,712 (56.14 MB)

In [5]:
# 5) Compile and set callbacks
initial_lr = 1e-4
model.compile(optimizer=Adam(learning_rate=initial_lr),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
checkpoint_path = "vgg16_head_best.h5"
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
]

In [7]:
# 6) Train only the head first (base frozen)
epochs_head = 5

history_head = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs_head,
    callbacks=callbacks
)

# Plot training curves for head training
def plot_history(h, title_suffix=""):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(h.history['accuracy'], label='train_acc')
    plt.plot(h.history['val_accuracy'], label='val_acc')
    plt.title('Accuracy ' + title_suffix)
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(h.history['loss'], label='train_loss')
    plt.plot(h.history['val_loss'], label='val_loss')
    plt.title('Loss ' + title_suffix)
    plt.legend()
    plt.show()

plot_history(history_head, "(head training)")

Epoch 1/5
 24/987 ━━━━━━━━━━━━━━━━━━━━ 50:45 3s/step - accuracy: 0.7748 - loss: 0.8672  

KeyboardInterrupt: 

In [ ]:
# 7) Fine-tuning: unfreeze top VGG blocks (e.g., last 4-5 conv layers) and recompile with lower LR
# Unfreeze some layers — common practice: unfreeze the top convolutional blocks, keep batchnorm behavior in mind
base_model.trainable = True

# Option: inspect layer names to choose cutoff
for i, layer in enumerate(base_model.layers):
    # print(i, layer.name, layer.trainable)
    pass

# Freeze lower layers, unfreeze last conv blocks (here we unfreeze from block5_conv1 onwards)
set_trainable = False
for layer in base_model.layers:
    if layer.name == "block5_conv1":
        set_trainable = True
    layer.trainable = set_trainable

# Recompile with a lower learning rate
fine_tune_lr = 1e-5
model.compile(optimizer=Adam(learning_rate=fine_tune_lr),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# 8) Fine-tune
epochs_finetune = 20
history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs_finetune,
    callbacks=callbacks
)

plot_history(history_ft, "(fine-tuning)")

In [ ]:
# 9) Final evaluation on validation set
# Predict (note: using val_gen with shuffle=False)
val_gen.reset()
pred_probs = model.predict(val_gen, verbose=1)
pred_classes = np.argmax(pred_probs, axis=1)
true_classes = val_gen.classes
labels = list(val_gen.class_indices.keys())

print("Classification Report:\n")
print(classification_report(true_classes, pred_classes, target_names=labels))

# Confusion matrix
cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# 10) Save final model
model.save("vgg16_emotion_model.h5")
print("Saved final model as vgg16_emotion_model.h5")

In [ ]:
# 11) (Optional) — Grad-CAM visualization for a few images
# Note: This is optional and may require small helpers. Uncomment and run if you'd like visual explanations.

"""
import cv2
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.cm as cm

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Example usage for one image path:
img_path = 'path_to_some_image.jpg'
img = image.load_img(img_path, target_size=img_size)
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)
heatmap = make_gradcam_heatmap(x, model, last_conv_layer_name='block5_conv3')
# Then overlay heatmap on original image using cv2 and matplotlib
"""

## Notes & tips
- If training is slow or GPU memory is limited, reduce `batch_size` or use fewer trainable layers for fine-tuning.
- If classes are imbalanced, consider computing `class_weight` from labels and pass to `.fit(..., class_weight=class_weight)`.
- Monitor `val_loss` and `val_accuracy` for overfitting; reduce LR or use stronger augmentation if needed.
- You can adjust which layers to unfreeze—commonly unfreeze block5 or block4+block5 depending on dataset size and similarity.